<a href="https://colab.research.google.com/github/gabrielcabanilla1/EXSU-500-Clinical-NLP-large-language-models-rct/blob/main/notebooks/03c_pubmedbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
DATA_VERSION = 'PubMed_20k_RCT'
PROJECT_DIR = '/content/drive/MyDrive/exsu_project'
DATA_DIR = f'{PROJECT_DIR}/prepared_data'
MODELS_DIR = f'{PROJECT_DIR}/models'
PREDICTIONS_DIR = f'{PROJECT_DIR}/predictions'
RESULTS_DIR = f'{PROJECT_DIR}/results'
CLASSES = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']
SEED = 42
N_BOOTSTRAP = 2000
EXPECTED_SIZES = {'train': 180040, 'dev': 30212}

VERSIONS = ['context', 'text']
RETRAIN = False
SUBSAMPLE = None
MODEL_NAME = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
EPOCHS = 2
LEARNING_RATE = 3e-5
VERSION_SETTINGS = {'context': {'use_context': True, 'batch_size': 32, 'max_length': 256},
                    'text': {'use_context': False, 'batch_size': 64, 'max_length': 128}}

def run_name(version):
    return f'pubmedbert_{version}' + (f'_sub{SUBSAMPLE}' if SUBSAMPLE else '')

print('Versions:', [run_name(v) for v in VERSIONS], '| retrain:', RETRAIN)

Versions: ['pubmedbert_context', 'pubmedbert_text'] | retrain: False


In [2]:
import os
import json
import time
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
for folder in (MODELS_DIR, PREDICTIONS_DIR, RESULTS_DIR):
    os.makedirs(folder, exist_ok=True)

def load_split(split):
    return pd.read_csv(f'{DATA_DIR}/{DATA_VERSION}_{split}.csv',
                       dtype={'abstract_id': str}, keep_default_na=False)

train, dev = load_split('train'), load_split('dev')
for name, df in (('train', train), ('dev', dev)):
    assert len(df) == EXPECTED_SIZES[name], f'Unexpected {name} size: re-run 02_data_preparation'
    assert set(df.label) <= set(CLASSES), f'Unknown labels in {name}'
assert not set(train.abstract_id) & set(dev.abstract_id), 'Leakage: abstract in train and dev'
print(f'Train: {train.abstract_id.nunique():,} abstracts / {len(train):,} sentences | '
      f'Dev: {dev.abstract_id.nunique():,} abstracts / {len(dev):,} sentences')

Mounted at /content/drive
Train: 15,000 abstracts / 180,040 sentences | Dev: 2,500 abstracts / 30,212 sentences


In [3]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, roc_auc_score,
                             average_precision_score, confusion_matrix)

def bootstrap_macro_f1(y_true, y_pred, abstract_ids, n_resamples=N_BOOTSTRAP, seed=SEED):
    index, unique_ids = pd.factorize(pd.Series(abstract_ids))
    k = len(CLASSES)
    counts = np.zeros((len(unique_ids), k, k), dtype=np.int64)
    np.add.at(counts, (index, y_true, y_pred), 1)
    rng = np.random.default_rng(seed)
    scores = np.empty(n_resamples)
    for b in range(n_resamples):
        cm = counts[rng.integers(0, len(unique_ids), len(unique_ids))].sum(axis=0)
        denominator = cm.sum(axis=0) + cm.sum(axis=1)
        scores[b] = np.divide(2 * np.diag(cm), denominator, out=np.zeros(k), where=denominator > 0).mean()
    return np.percentile(scores, [2.5, 97.5])

def evaluate_and_save(run_name, model_description, input_description, df, probs,
                      fit_seconds, hardware, settings):
    probs = np.asarray(probs, dtype=float)
    assert probs.shape == (len(df), len(CLASSES)), 'probs must be (n_sentences, 5) in CLASSES order'
    k = len(CLASSES)
    y_true = pd.Categorical(df.label, categories=CLASSES).codes
    y_pred = probs.argmax(axis=1)
    one_hot = np.eye(k)[y_true]

    cm = confusion_matrix(y_true, y_pred, labels=range(k))
    tp = np.diag(cm); fn = cm.sum(axis=1) - tp; fp = cm.sum(axis=0) - tp; tn = cm.sum() - tp - fn - fp
    per_class = pd.DataFrame({
        'precision': precision_score(y_true, y_pred, labels=range(k), average=None, zero_division=0),
        'sensitivity': tp / (tp + fn),
        'specificity': tn / (tn + fp),
        'f1': f1_score(y_true, y_pred, labels=range(k), average=None, zero_division=0),
        'auroc': [roc_auc_score(one_hot[:, i], probs[:, i]) for i in range(k)],
        'auprc': [average_precision_score(one_hot[:, i], probs[:, i]) for i in range(k)],
        'prevalence': one_hot.mean(axis=0),
    }, index=pd.Index(CLASSES, name='class'))
    per_class.loc['macro average'] = per_class.mean()

    lower, upper = bootstrap_macro_f1(y_true, y_pred, df.abstract_id)
    summary = {
        'run': run_name, 'model': model_description, 'input': input_description,
        'split': 'dev', 'abstracts': df.abstract_id.nunique(), 'sentences': len(df),
        'decision_rule': 'argmax',
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=range(k), average='macro', zero_division=0),
        'macro_f1_ci95_lower': lower, 'macro_f1_ci95_upper': upper,
        'weighted_f1': f1_score(y_true, y_pred, labels=range(k), average='weighted', zero_division=0),
        'macro_auroc': per_class.loc['macro average', 'auroc'],
        'macro_auprc': per_class.loc['macro average', 'auprc'],
        'macro_sensitivity': per_class.loc['macro average', 'sensitivity'],
        'macro_specificity': per_class.loc['macro average', 'specificity'],
        'fit_seconds': round(float(fit_seconds), 1), 'hardware': hardware,
        'seed': SEED, 'n_bootstrap': N_BOOTSTRAP, 'settings': json.dumps(settings),
    }

    predictions = df[['abstract_id', 'sentence_idx', 'label']].copy()
    predictions['prediction'] = np.array(CLASSES)[y_pred]
    for i, c in enumerate(CLASSES):
        predictions[f'prob_{c}'] = probs[:, i]
    predictions.to_csv(f'{PREDICTIONS_DIR}/dev_{run_name}.csv', index=False)
    pd.DataFrame([summary]).to_csv(f'{RESULTS_DIR}/dev_{run_name}_summary.csv', index=False)
    per_class.to_csv(f'{RESULTS_DIR}/dev_{run_name}_per_class.csv')
    confusion = pd.DataFrame(cm, index=pd.Index(CLASSES, name='actual'),
                             columns=pd.Index(CLASSES, name='predicted'))
    confusion.to_csv(f'{RESULTS_DIR}/dev_{run_name}_confusion_matrix.csv')

    print(f"{model_description} | {input_description}")
    print(f"Dev macro-F1 {summary['macro_f1']:.4f} (95% CI {lower:.4f}-{upper:.4f}) | "
          f"accuracy {summary['accuracy']:.4f} | weighted F1 {summary['weighted_f1']:.4f} | "
          f"macro AUROC {summary['macro_auroc']:.4f} | macro AUPRC {summary['macro_auprc']:.4f}")
    print(f'Saved predictions and 3 result tables for run "{run_name}"')
    print('\nPer-class metrics (one class vs. the other four; decision rule: argmax):')
    display(per_class.round(4))
    print('\nConfusion matrix (rows = actual class, columns = predicted class):')
    display(confusion)
    never_predicted = [c for i, c in enumerate(CLASSES) if (y_pred == i).sum() == 0]
    print('Classes never predicted:', never_predicted or 'none')
    return summary, per_class

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '|', torch.cuda.get_device_name(0) if device == 'cuda' else 'NO GPU')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenise(df, use_context, max_length):
    if use_context:
        return tokenizer(list(df.text), list(df.context), truncation='longest_first',
                         max_length=max_length, padding='max_length', return_tensors='pt')
    return tokenizer(list(df.text), truncation=True, max_length=max_length,
                     padding='max_length', return_tensors='pt')

def as_dataset(tokens, labels=None):
    tensors = [tokens['input_ids'], tokens['attention_mask']]
    if labels is not None:
        tensors.append(torch.tensor(labels))
    return TensorDataset(*tensors)

@torch.no_grad()
def predict(model, dataset):
    model.eval()
    probs = []
    for ids, mask in DataLoader(dataset, batch_size=256):
        with torch.autocast(device, enabled=(device == 'cuda')):
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(probs)

Device: cuda | NVIDIA L4


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

In [5]:
def train_version(version, dev_dataset):
    settings = VERSION_SETTINGS[version]
    model_dir = f'{MODELS_DIR}/{run_name(version)}'
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    train_run = train.sample(SUBSAMPLE, random_state=SEED) if SUBSAMPLE else train
    tokens_train = tokenise(train_run, settings['use_context'], settings['max_length'])
    train_loader = DataLoader(as_dataset(tokens_train, train_run.y.values), batch_size=settings['batch_size'],
                              shuffle=True, generator=torch.Generator().manual_seed(SEED))
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(CLASSES)).to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimiser, int(0.06 * total_steps), total_steps)
    scaler = torch.amp.GradScaler(enabled=(device == 'cuda'))
    print(f'{run_name(version)}: training on {len(train_run):,} sentences')

    best_f1, start, log = -1, time.time(), []
    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, y_batch in tqdm(train_loader, desc=f'{version} epoch {epoch + 1}/{EPOCHS}'):
            optimiser.zero_grad()
            with torch.autocast(device, enabled=(device == 'cuda')):
                loss = model(input_ids=ids.to(device), attention_mask=mask.to(device),
                             labels=y_batch.to(device)).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimiser)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimiser)
            scaler.update()
            scheduler.step()
        dev_f1 = f1_score(dev.y, predict(model, dev_dataset).argmax(axis=1), average='macro')
        minutes = (time.time() - start) / 60
        log.append({'epoch': epoch + 1, 'dev_macro_f1': round(dev_f1, 4), 'minutes': round(minutes, 1)})
        print(f'Epoch {epoch + 1}: dev macro-F1 = {dev_f1:.4f} | {minutes:.1f} min')
        if dev_f1 > best_f1:
            best_f1 = dev_f1
            model.save_pretrained(model_dir)
            tokenizer.save_pretrained(model_dir)
            print('  -> best so far, saved to Drive')
    run_info = {'run': run_name(version), 'model': MODEL_NAME, 'use_context': settings['use_context'],
                'subsample': SUBSAMPLE, 'epochs': EPOCHS, 'batch_size': settings['batch_size'],
                'max_length': settings['max_length'], 'learning_rate': LEARNING_RATE, 'seed': SEED,
                'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else 'cpu',
                'total_minutes': round((time.time() - start) / 60, 1), 'log': log}
    json.dump(run_info, open(f'{model_dir}/run_info.json', 'w'), indent=2)
    del model, optimiser, train_loader, tokens_train
    if device == 'cuda':
        torch.cuda.empty_cache()

In [6]:
summaries = {}
for version in VERSIONS:
    settings = VERSION_SETTINGS[version]
    model_dir = f'{MODELS_DIR}/{run_name(version)}'
    print('=' * 100, f'\n{run_name(version)}')
    tokens_dev = tokenise(dev, settings['use_context'], settings['max_length'])
    dev_dataset = as_dataset(tokens_dev)
    if RETRAIN or not os.path.exists(f'{model_dir}/run_info.json'):
        train_version(version, dev_dataset)
    else:
        print('Already trained: reusing the model saved in', model_dir)
    run_info = json.load(open(f'{model_dir}/run_info.json'))
    print('Training log:', run_info['log'], '|', run_info['total_minutes'], 'min on', run_info['gpu'])

    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    dev_probs = predict(model, dev_dataset)
    best_epoch = max(run_info['log'], key=lambda r: r['dev_macro_f1'])['epoch']
    summaries[version], _ = evaluate_and_save(
        run_name(version), 'Fine-tuned PubMedBERT',
        'sentence + position + neighbours' if settings['use_context'] else 'sentence only',
        dev, dev_probs, run_info['total_minutes'] * 60, run_info['gpu'],
        {'model_name': MODEL_NAME, 'epochs': EPOCHS, 'best_epoch': best_epoch,
         'batch_size': settings['batch_size'], 'max_length': settings['max_length'],
         'learning_rate': LEARNING_RATE, 'subsample': SUBSAMPLE})
    del model
    if device == 'cuda':
        torch.cuda.empty_cache()

print('=' * 100)
display(pd.DataFrame(summaries).T[['run', 'input', 'macro_f1', 'macro_f1_ci95_lower', 'macro_f1_ci95_upper',
                                   'accuracy', 'macro_auroc', 'macro_auprc']])

pubmedbert_context


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

pubmedbert_context: training on 180,040 sentences


... (30 outputs left out)


Confusion matrix (rows = actual class, columns = predicted class):


predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
actual,,,,,
BACKGROUND,2893,314,96,12,134
OBJECTIVE,862,1400,66,3,45
METHODS,92,46,9578,221,27
RESULTS,21,1,358,9134,327
CONCLUSIONS,296,7,53,338,3888


Classes never predicted: none


,run,input,macro_f1,macro_f1_ci95_lower,macro_f1_ci95_upper,accuracy,macro_auroc,macro_auprc
context,pubmedbert_context,sentence + position + neighbours,0.885004,0.878676,0.891152,0.932643,0.991349,0.938921
text,pubmedbert_text,sentence only,0.83724,0.830897,0.843709,0.890143,0.982234,0.901062


In [7]:
RUN_NAME = run_name('context')
predictions = pd.read_csv(f'{PREDICTIONS_DIR}/dev_{RUN_NAME}.csv', dtype={'abstract_id': str}, keep_default_na=False)
review = predictions.merge(dev[['abstract_id', 'sentence_idx', 'n_sentences', 'text', 'previous', 'next']],
                           on=['abstract_id', 'sentence_idx'], validate='one_to_one')
probs = review[[f'prob_{c}' for c in CLASSES]].to_numpy()
review['prob_true'] = probs[np.arange(len(review)), review.label.map(CLASSES.index)]
review['prob_predicted'] = probs.max(axis=1)
errors = review[review.label != review.prediction].copy()
errors['confusion'] = errors.label + ' -> ' + errors.prediction
errors['sentence_position'] = (errors.sentence_idx + 1).astype(str) + ' of ' + errors.n_sentences.astype(str)
errors['review_note'] = ''
errors = errors.sort_values(['confusion', 'prob_predicted'], ascending=[True, False])[
    ['confusion', 'abstract_id', 'sentence_position', 'label', 'prediction', 'prob_true', 'prob_predicted',
     'previous', 'text', 'next', 'review_note']].round(3)
errors.to_csv(f'{PREDICTIONS_DIR}/errors_dev_{RUN_NAME}.csv', index=False)
print(f'{len(errors):,} misclassified sentences saved for review. Errors by type:')
print(errors.confusion.value_counts().head(8).to_string())

2,035 misclassified sentences saved for review. Errors by type:
confusion
OBJECTIVE -> BACKGROUND    841
BACKGROUND -> OBJECTIVE    402
RESULTS -> METHODS         215
METHODS -> RESULTS         157
RESULTS -> CONCLUSIONS     109
CONCLUSIONS -> RESULTS     106
METHODS -> BACKGROUND       40
BACKGROUND -> METHODS       40


In [8]:
predictions = predictions.sort_values(['abstract_id', 'sentence_idx']).reset_index(drop=True)
probs = predictions[[f'prob_{c}' for c in CLASSES]].to_numpy()
y_true = predictions.label.map(CLASSES.index).to_numpy()
macro = lambda y, p: f1_score(y, p, labels=range(len(CLASSES)), average='macro', zero_division=0)

k = len(CLASSES)
start_counts, trans_counts = np.ones(k), np.ones((k, k))
for _, labels in train.sort_values(['abstract_id', 'sentence_idx']).groupby('abstract_id', sort=False).y:
    labels = labels.to_numpy()
    start_counts[labels[0]] += 1
    np.add.at(trans_counts, (labels[:-1], labels[1:]), 1)
log_start = np.log(start_counts / start_counts.sum())
log_trans = np.log(trans_counts / trans_counts.sum(axis=1, keepdims=True))
log_probs = np.log(np.clip(probs, 1e-9, 1))
abstract_rows = list(predictions.groupby('abstract_id', sort=False).indices.values())

def viterbi_decode(weight):
    out = np.empty(len(predictions), dtype=int)
    for rows in abstract_rows:
        lp = log_probs[rows]
        score, back = lp[0] + weight * log_start, np.zeros((len(rows), k), dtype=int)
        for t in range(1, len(rows)):
            candidates = score[:, None] + weight * log_trans
            back[t] = candidates.argmax(axis=0)
            score = candidates.max(axis=0) + lp[t]
        path = [score.argmax()]
        for t in range(len(rows) - 1, 0, -1):
            path.append(back[t, path[-1]])
        out[rows] = path[::-1]
    return out

def objective_boost(factor):
    adjusted = probs.copy()
    adjusted[:, CLASSES.index('OBJECTIVE')] *= factor
    return adjusted.argmax(axis=1)

rng = np.random.default_rng(SEED)
half_A = set(rng.choice(predictions.abstract_id.unique(), predictions.abstract_id.nunique() // 2, replace=False))
in_A = predictions.abstract_id.isin(half_A).to_numpy()
for name, decode, grid in [('Viterbi weight', viterbi_decode, [0.0, 0.25, 0.5, 1.0, 1.5]),
                           ('OBJECTIVE factor', objective_boost, [1.0, 1.25, 1.5, 2.0, 3.0])]:
    decoded = {v: decode(v) for v in grid}
    for label, choose, measure in [('A -> B', in_A, ~in_A), ('B -> A', ~in_A, in_A)]:
        best = max(grid, key=lambda v: macro(y_true[choose], decoded[v][choose]))
        print(f'{name:17s} {label}: chosen {best:<5} | macro-F1 on the other half '
              f'{macro(y_true[measure], decoded[grid[0]][measure]):.4f} -> {macro(y_true[measure], decoded[best][measure]):.4f}')

Viterbi weight    A -> B: chosen 0.25  | macro-F1 on the other half 0.8879 -> 0.8889
Viterbi weight    B -> A: chosen 0.25  | macro-F1 on the other half 0.8820 -> 0.8826
OBJECTIVE factor  A -> B: chosen 1.5   | macro-F1 on the other half 0.8879 -> 0.8883
OBJECTIVE factor  B -> A: chosen 1.25  | macro-F1 on the other half 0.8820 -> 0.8825
